### This notebook is used for small-scale PRAS simulation to run and save data. For full-scale simulation, please refer to the pras_simulation Julia script.
* It read the Matpower data file that has been constructed from the Security Constraint multi-network run from 2024 to 2044. The last network (2044) is constructed and generates the file (nw_2044_prod_v1.m). This is used as the base data file, and on top of that, storage and generators are added.
* Time series data with thermal generators' expiry dates are implemented.
* Future storage and generation with time series data added.
* Storage time series is not implemented in the PowerSystems; the time series data with binary 0 and 1, indicating the power value with the starting time (1) and multiplying with the actual power, is directly added to the pras storage matrix data.
* Every expiry time is hard-coded with the 1st of Dec with the respective year added for expiry thermal generators, and similarly for storage time series data.
* It has an additional 23 buses and has an additional 18 new "Connection "  lines

In [1]:
using PowerSystems
using CSV
#using HDF5
using DataFrames
using Dates
using PRAS
using JuMP
#using Plots
using PowerModels
using PowerSimulations
using PowerSystemCaseBuilder
#using PowerNetworkMatrices
using StorageSystemsSimulations
#using HiGHS
using TimeSeries
using Dates: DateTime
using Logging
using DataStructures

import InfrastructureSystems
const IS = InfrastructureSystems
#import InfrastructureModels as _IM

Logging.disable_logging(Logging.Warn)
PowerModels.silence()

import DataStructures: SortedDict
const PSY = PowerSystems
#const PSI = PowerSimulations
#const PSB = PowerSystemCaseBuilder
#const PNM = PowerNetworkMatrices

ENV["TMPDIR"] = "/tmp"


[info | PowerModels]: Suppressing information and warning messages for the rest of this session.  Use the Memento package for more fine-grained control of logging.


"/tmp"

In [25]:
include("src/utility.jl")
include("src/pras_utility.jl")

save_pras_lines_limits_info (generic function with 1 method)

In [37]:
file_path = joinpath(pwd(), "data", "sc_data", "nw_2044_prod_v1.m")
sys, pm_data, base_storage_data = initialise_system_v3(file_path)
println("system loaded")

┌ Error: Generator voltage set-points for bus 1032 are inconsistent. This can lead to unexpected results
└ @ PowerSystems /datasets/work/en-energy-sys/work/users/bal246/JULIA_EV_1111/packages/PowerSystems/AHyDB/src/parsers/pm_io/matpower.jl:245
┌ Error: Generator voltage set-points for bus 933 are inconsistent. This can lead to unexpected results
└ @ PowerSystems /datasets/work/en-energy-sys/work/users/bal246/JULIA_EV_1111/packages/PowerSystems/AHyDB/src/parsers/pm_io/matpower.jl:245


system loaded


┌ Error: Generator voltage set-points for bus 25 are inconsistent. This can lead to unexpected results
└ @ PowerSystems /datasets/work/en-energy-sys/work/users/bal246/JULIA_EV_1111/packages/PowerSystems/AHyDB/src/parsers/pm_io/matpower.jl:245
┌ Error: Generator voltage set-points for bus 1544 are inconsistent. This can lead to unexpected results
└ @ PowerSystems /datasets/work/en-energy-sys/work/users/bal246/JULIA_EV_1111/packages/PowerSystems/AHyDB/src/parsers/pm_io/matpower.jl:245
┌ Error: Generator voltage set-points for bus 1544 are inconsistent. This can lead to unexpected results
└ @ PowerSystems /datasets/work/en-energy-sys/work/users/bal246/JULIA_EV_1111/packages/PowerSystems/AHyDB/src/parsers/pm_io/matpower.jl:245
┌ Error: Generator voltage set-points for bus 1544 are inconsistent. This can lead to unexpected results
└ @ PowerSystems /datasets/work/en-energy-sys/work/users/bal246/JULIA_EV_1111/packages/PowerSystems/AHyDB/src/parsers/pm_io/matpower.jl:245
┌ Error: Generator vol

In [17]:
#add_future_gen(sys)
#add_future_storage(sys)

In [18]:
scenario = "MT"
location = joinpath(pwd(), "data", "sc_data")
file_path = joinpath(location, "future_gen_thermal_exp_pp.csv")
add_future_status_therm_stor(sys, scenario, file_path, "gen")

file_path = joinpath(location, "future_storage_pp.csv")
add_future_status_therm_stor(sys, scenario, file_path, "storage")
println("Added thermal generators and storage time series data")

Added thermal generators and storage time series data


In [126]:
#xx = get_component(ThermalStandard, sys, "gen_3442_1")
#xx
#xx = get_component(ThermalStandard, sys, "gen_1082_6")
#xx = get_component(EnergyReservoirStorage, sys, "Coordinated_CER_storage_15")
#aa = get_time_series_array(
#            SingleTimeSeries,
#            xx,
#            "max_active_power",)
#aa = get_time_series(SingleTimeSeries, xx, "max_active_power")
#aa.data
#println("done")

In [44]:
scenario = "MT"
location = joinpath(pwd(), "data", "sc_data")
file_path = joinpath(location, "future_gen_thermal_exp_pp.csv")
add_future_gen(sys, scenario, location)
add_future_storage(sys, scenario, location)
add_future_status_therm_stor(sys, scenario, file_path, "gen")
file_path = joinpath(location, "future_storage_pp.csv")
add_future_status_therm_stor(sys, scenario, file_path, "storage")

if scenario == "ST"
    time_series_data = built_load_one_yearly_TSdata(sys)
elseif scenario == "MT"
    time_series_data = built_load_six_yearly_TSdata(sys)    
elseif scenario == "LT"
    time_series_data = built_load_twenty_years_TSdata(sys)
elseif scenario == "SUM_ED"
    time_series_data = built_load_SUM_ED_TSdata(sys)
end
println("Added thermal generators and storage time series data")

Added thermal generators and storage time series data


In [24]:
#save_newly_built_system(sys, scenario)

In [40]:
using SiennaPRASInterface

const DEFAULT_DEVICE_MODELS_ST = [
        DeviceRAModel(PSY.Line, LinePRAS),
        DeviceRAModel(PSY.TwoTerminalHVDCLine, LinePRAS),
        DeviceRAModel(PSY.StaticLoad, StaticLoadPRAS),
        DeviceRAModel(PSY.ThermalGen, GeneratorPRAS),
        DeviceRAModel(PSY.RenewableGen, GeneratorPRAS),
        DeviceRAModel(PSY.HydroDispatch, GeneratorPRAS),
        DeviceRAModel(PSY.EnergyReservoirStorage, EnergyReservoirLossless)
    ]   
const DEFAULT_TEMPLATE = RATemplate(PSY.Area, DEFAULT_DEVICE_MODELS_ST)

gps = generate_pras_system(sys, DEFAULT_TEMPLATE)
println("PRAS model - GPS - is created for scenario:", scenario)

PRAS model - GPS - is created for scenario:MT

In [41]:
# This is required to add time series storage data here, as importing the scaling values in the
# Power Systems not being implemented
for (i, kk) in enumerate(gps.storages.names)
    comp_data = get_component(EnergyReservoirStorage, sys, kk)
    ts_data = get_time_series(SingleTimeSeries, comp_data, "max_active_power")
    ts_data = ts_data.data
    gps.storages.energy_capacity[i,:] = gps.storages.energy_capacity[i,:] .* values(ts_data)
    gps.storages.charge_capacity[i,:] = gps.storages.charge_capacity[i,:] .* values(ts_data)
    gps.storages.discharge_capacity[i,:] = gps.storages.discharge_capacity[i,:] .* values(ts_data)
end
println("done")

done


In [42]:
resultspecs = (Shortfall(), Surplus(), Flow(), Utilization(), StorageEnergy(),
    GeneratorStorageEnergy(),GeneratorAvailability(), LineAvailability(), StorageAvailability(),
    GeneratorStorageAvailability())
samples = 100
println("scenario: ", scenario,",", "sample size: ", samples)
println("SMC starting at ", now())
smallsample = SequentialMonteCarlo(samples=samples, seed=123, threaded=false)
shortfall_rs, surplus_rs, flow_rs, util_rs, energy, gs_energy, ga, la, sa, gsa =
        assess(gps, smallsample, resultspecs...)
println("PRAS simulation done for sample size:", samples,",", now())
odir = joinpath(pwd(), "data", "pras_metrics_output", "shortfall_data", scenario)
saveshortfall(shortfall_rs, gps, odir)
println("PRAS result is saved")

scenario: MT,sample size: 100
SMC starting at 2026-03-27T21:03:38.506
PRAS simulation done for sample size:100,2026-03-27T21:13:58.821
PRAS result is saved


In [43]:
odir = joinpath(pwd(), "data", "pras_metrics_output", "shortfall_data", scenario)
use_df = save_shortfall_eue_metrics(gps, shortfall_rs, scenario, odir)
println("Unserved energy during shortfall time extracted")
save_shortfall_time_series_data(scenario, shortfall_rs, gps, flow_rs, util_rs, odir)
save_generator_storage_data(ga, gps, energy, scenario, odir)

done
Unserved energy during shortfall time extracted
Saved shortfall data
pras time series metrics calculated
Saved PRAS time series metrics data
pras regional loads calculated
pras metrics data saved
Saved energy data
